In [2]:
import sys
print(sys.executable)
import pandas as pd
import numpy as np
import pickle
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1. CHARGEMENT DU FICHIER CSV
# ─────────────────────────────────────────────
FILE_PATH = "imputation.csv"   #
OUTPUT_PATH = "resultats_avec_predictionsimputation.csv"
MODEL_PATH = "model_groupe.pkl"

print("=" * 60)
print("  PIPELINE ML - PRÉDICTION GROUPE")
print("=" * 60)

df = pd.read_csv(FILE_PATH, sep=';', encoding='utf-8', low_memory=False)
print(f"\n[1] Fichier chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

# ─────────────────────────────────────────────
# 2. NETTOYAGE DU FICHIER 
# ─────────────────────────────────────────────
print("\n[2] Nettoyage en cours...")

FEATURES = ['DOMAINE', 'SOUS-DOMAINE', 'TYPE', 'TYPOLOGIE'] # Element pour la prediction 
TARGET   = 'GROUPE'# La prediction ce fait au niveau du groupe 

# Normalisation des noms de colonnes (trim, uppercase)
df.columns = [c.strip().upper() for c in df.columns]

# Fiare un Trim + uppercase sur les colonnes texte clés
for col in FEATURES + [TARGET]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = df[col].replace({'NAN': np.nan, '': np.nan, 'NONE': np.nan})

# je m'assure de supprimer les doublons exacts
before = len(df) 
df.drop_duplicates(inplace=True)
print(f"   → Doublons supprimés : {before - len(df)}")

# Faire en sorte d'Afficher les valeurs manquantes
print("\n   Valeurs manquantes par colonne clé :")
for col in FEATURES + [TARGET]:
    if col in df.columns:
        n = df[col].isna().sum()
        pct = 100 * n / len(df)
        print(f"     {col:<20} {n:>5} ({pct:.1f}%)")

# ─────────────────────────────────────────────
# 3. PRÉPARATION DES DONNÉES
# ─────────────────────────────────────────────
print("\n[3] Préparation des données...")

# Séparer les lignes avec/sans GROUPE
df_labeled   = df[df[TARGET].notna()].copy()
df_unlabeled = df[df[TARGET].isna()].copy()

print(f"   → Lignes avec GROUPE (train/test) : {len(df_labeled)}")
print(f"   → Lignes sans GROUPE (à prédire)  : {len(df_unlabeled)}")
# Supprimer les classes avec trop peu d'exemples (< 2) → impossible à splitter
class_counts = df_labeled[TARGET].value_counts()
valid_classes = class_counts[class_counts >= 2].index
removed = df_labeled[~df_labeled[TARGET].isin(valid_classes)][TARGET].unique()
if len(removed) > 0:
    print(f"   → Classes retirées (< 2 exemples) : {list(removed)}")
df_labeled = df_labeled[df_labeled[TARGET].isin(valid_classes)]

print(f"\n   Classes GROUPE disponibles ({df_labeled[TARGET].nunique()}) :")
for cls, cnt in df_labeled[TARGET].value_counts().items():
    print(f"     {cls:<35} {cnt:>4}")

# Encodage
encoders = {}
X = df_labeled[FEATURES].copy()
for col in FEATURES:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].fillna('INCONNU').astype(str))
    encoders[col] = le

le_target = LabelEncoder()
y = le_target.fit_transform(df_labeled[TARGET])
encoders[TARGET] = le_target

# ─────────────────────────────────────────────
# 4. ENTRAÎNEMENT DE PLUSIEURS MODÈLES
# ─────────────────────────────────────────────

print("\n[4] Entraînement de plusieurs modèles...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   → Train : {len(X_train)} | Test : {len(X_test)}")

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    
    "LogisticRegression": LogisticRegression(
        max_iter=2000
    ),
    
    "DecisionTree": DecisionTreeClassifier(
        random_state=42
    ),
    
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42
    )
}

results = {}

for name, model in models.items():
    
    print(f"\n--- Entraînement : {name} ---")
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    results[name] = acc
    
    print(f"Accuracy : {acc:.2%}")

    print("\n==============================")
print("Comparaison des modèles")
print("==============================")

for model_name, score in results.items():
    print(f"{model_name:<20} {score:.2%}")

best_model_name = max(results, key=results.get)
print(f"\n Meilleur modèle : {best_model_name}")

best_model = models[best_model_name]
model = best_model
# ─────────────────────────────────────────────
# 5. FAIRE UNE ÉVALUATION SUR LE JEU DU TEST
# ─────────────────────────────────────────────
print("\n[5] Évaluation sur le jeu de test...")

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n Accuracy globale : {acc:.1%}")

# Classes présentes dans le test
labels_in_test = np.unique(np.concatenate([y_test, y_pred]))
class_names = le_target.inverse_transform(labels_in_test)

print("\n   Rapport de classification :")
print(classification_report(
    y_test, y_pred,
    labels=labels_in_test,
    target_names=class_names
))

# Importance des features
print("   Importance des variables :")
for feat, imp in sorted(zip(FEATURES, model.feature_importances_), key=lambda x: -x[1]):
    bar = "█" * int(imp * 40)
    print(f"     {feat:<20} {imp:.3f}  {bar}")

# Graphique importance
fig, ax = plt.subplots(figsize=(7, 4))
feat_imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
feat_imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title("Importance des variables")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("importance_features.png", dpi=150)
plt.close()
print("\n  Graphique sauvegardé : importance_features.png")

# ─────────────────────────────────────────────
# 6. PRÉDICTION SUR LES LIGNES SANS GROUPE
# ─────────────────────────────────────────────
print(f"\n[6] Prédiction sur {len(df_unlabeled)} lignes sans GROUPE...")

if len(df_unlabeled) > 0:
    X_pred = df_unlabeled[FEATURES].copy()

    # Encoder avec les encodeurs existants, gérer les valeurs inconnues
    for col in FEATURES:
        le = encoders[col]
        known_classes = set(le.classes_)
        X_pred[col] = X_pred[col].fillna('INCONNU').astype(str)
        X_pred[col] = X_pred[col].apply(
            lambda v: v if v in known_classes else le.classes_[0]
        )
        X_pred[col] = le.transform(X_pred[col])

    preds = model.predict(X_pred)
    probas = model.predict_proba(X_pred).max(axis=1)

    df_unlabeled = df_unlabeled.copy()
    df_unlabeled[TARGET] = le_target.inverse_transform(preds)
    df_unlabeled['GROUPE_CONFIANCE'] = (probas * 100).round(1).astype(str) + '%'

    print("   Répartition des prédictions :")
    for cls, cnt in df_unlabeled[TARGET].value_counts().items():
        print(f"     {cls:<35} {cnt:>4}")
else:
    print("   → Aucune ligne à prédire.")

# ─────────────────────────────────────────────
# 7. EXPORT
# ─────────────────────────────────────────────
print(f"\n[7] Export du fichier final...")

# Réassemblage : lignes connues + lignes prédites
df_labeled_out = df_labeled.copy()
df_labeled_out['GROUPE_CONFIANCE'] = 'Réel'

if len(df_unlabeled) > 0:
    df_final = pd.concat([df_labeled_out, df_unlabeled], ignore_index=True)
else:
    df_final = df_labeled_out

df_final.to_csv(OUTPUT_PATH, sep=';', index=False, encoding='utf-8-sig')
print(f"  Fichier exporté : {OUTPUT_PATH}")

# Sauvegarder le modèle
pickle.dump({'model': model, 'encoders':  encoders, 'features': FEATURES}, open(MODEL_PATH, 'wb'))
print(f" Modèle sauvegardé : {MODEL_PATH}")

print("\n" + "=" * 60)
print("  PIPELINE TERMINÉ")
print("=" * 60)


# ─────────────────────────────────────────────
# BONUS A VOIR  : Fonction de prédiction rapide
# ─────────────────────────────────────────────
def predire_groupe(domaine, sous_domaine, type_, typologie, model_path=MODEL_PATH):
    """
    Prédit le GROUPE pour un ticket donné.
    
    Exemple :
        predire_groupe('MOBILE', 'RADIO_MOBILE', 'PLAINTES GP', 'COUVERTURE')
    """
    bundle = pickle.load(open(model_path, 'rb'))
    m, enc, feats = bundle['model'], bundle['encoders'], bundle['features']    
    vals = {
        'DOMAINE'      : domaine.strip().upper(),
        'SOUS-DOMAINE' : sous_domaine.strip().upper(),
        'TYPE'         : type_.strip().upper(),
        'TYPOLOGIE'    : typologie.strip().upper(),
    }
    
    row = []
    for col in feats:
        le = enc[col]
        v  = vals[col]
        if v not in le.classes_:
            v = le.classes_[0]  # fallback valeur inconnue
        row.append(le.transform([v])[0])
    
    pred  = m.predict([row])[0]
    proba = m.predict_proba([row])[0].max()
    groupe = enc[TARGET].inverse_transform([pred])[0]
    
    print(f"  GROUPE prédit : {groupe}  (confiance : {proba:.1%})")
    return groupe, proba


# Exemple d'utilisation :
# predire_groupe('MOBILE', 'RADIO_MOBILE', 'PLAINTES GP', 'COUVERTURE')

def predire_groupe(domaine, sous_domaine, type_, typologie, model_path="model_groupe.pkl"):
    import pickle
    bundle = pickle.load(open(model_path, 'rb'))
    m, enc, feats = bundle['model'], bundle['encoders'], bundle['features']
    
    vals = {
        'DOMAINE': domaine.strip().upper(),
        'SOUS-DOMAINE': sous_domaine.strip().upper(),
        'TYPE': type_.strip().upper(),
        'TYPOLOGIE': typologie.strip().upper()
    }
    
    row = []
    for col in feats:
        le = enc[col]
        v = vals[col]
        if v not in le.classes_:
            v = le.classes_[0]
        row.append(le.transform([v])[0])
    
    pred = m.predict([row])[0]
    proba = m.predict_proba([row])[0].max()
    groupe = enc["GROUPE"].inverse_transform([pred])[0]
    
    return groupe, proba





/Users/macbook/Desktop/Stage-premieretache/venv/bin/python
  PIPELINE ML - PRÉDICTION GROUPE

[1] Fichier chargé : 3639 lignes, 24 colonnes

[2] Nettoyage en cours...
   → Doublons supprimés : 0

   Valeurs manquantes par colonne clé :
     DOMAINE                 46 (1.3%)
     SOUS-DOMAINE            44 (1.2%)
     TYPE                    41 (1.1%)
     TYPOLOGIE               45 (1.2%)
     GROUPE                   0 (0.0%)

[3] Préparation des données...
   → Lignes avec GROUPE (train/test) : 3639
   → Lignes sans GROUPE (à prédire)  : 0

   Classes GROUPE disponibles (18) :
     DEX/ISOM/EIS                         985
     BAYE SALIOU SECK                     558
     HUAWEI                               380
     ACTIVATION                           320
     DI/IRM/IDR                           274
     DI/DAM/DAR                           172
     OUSMANE SANE                         169
     DEX/ISOM/SMO                         147
     DI/IRM/OPR                           121
